# Phase 3: XAI Audit Notebook

Deep-dive review of SHAP values explicitly extracted from the P50 median model.

In [ ]:
import pandas as pd
import numpy as np
import shap
import joblib
import matplotlib.pyplot as plt
from pathlib import Path
import sys
sys.path.insert(0, str(Path.cwd().parent))

from preprocessing.validator import validate_dataset
from preprocessing.title_standardiser import standardise_titles
from preprocessing.feature_engineer import engineer_features
from models.train_models import DATA_FILE, SEED, FEATURES, CATEGORICALS
from sklearn.model_selection import train_test_split

df_clean = standardise_titles(validate_dataset(str(DATA_FILE)).dataframe)
X = engineer_features(df_clean)[FEATURES]
for cat in CATEGORICALS: X[cat] = X[cat].astype('category')
_, X_test = train_test_split(X, test_size=0.2, random_state=SEED)

model_p50 = joblib.load('../models/saved/lgb_p50.pkl')
explainer = shap.TreeExplainer(model_p50)
shap_values = explainer(X_test)

## Career Velocity Dependence Plot

Confirming the non-linear U-shape or threshold effect.

In [ ]:
shap.plots.scatter(shap_values[:, "career_velocity"], color=shap_values[:, "job_level"])

## Individual Personas (Waterfall Plots)

In [ ]:
# Random male
male_idx = np.where(X_test["gender"] == "Male")[0][0]
shap.plots.waterfall(shap_values[male_idx])

In [ ]:
# Random female
female_idx = np.where(X_test["gender"] == "Female")[0][0]
shap.plots.waterfall(shap_values[female_idx])